# Exploratory Data Analysis
Home Credit portfolio: target balance, missingness, key risk drivers, and the pipeline's
stability, roll-rate and vintage outputs. Run the pipeline first (`python main.py run-all`);
paths are relative to the project root.

In [ ]:
import os
import polars as pl

os.chdir("..") if os.path.basename(os.getcwd()) == "notebooks" else None
P = "data/processed"
pl.Config.set_tbl_rows(25)
mar = pl.read_parquet(f"{P}/master_analytical_record.parquet")
mar.shape

## Target balance

In [ ]:
mar.select(pl.len().alias("applications"), pl.col("TARGET").mean().alias("default_rate"), pl.col("TARGET").sum().alias("defaults"))

## Missingness (top 15 columns)

In [ ]:
(mar.null_count().transpose(include_header=True, column_names=["nulls"])
    .with_columns((pl.col("nulls") / mar.height).alias("null_share"))
    .sort("null_share", descending=True).head(15))

## Default rate by segment

In [ ]:
def default_by(col, top=10):
    return (mar.group_by(col).agg(pl.len().alias("n"), pl.col("TARGET").mean().alias("default_rate"))
               .filter(pl.col("n") >= 1000).sort("default_rate", descending=True).head(top))

default_by("NAME_INCOME_TYPE")

In [ ]:
default_by("NAME_EDUCATION_TYPE")

## Engineered bureau and installment features vs default

In [ ]:
(mar.with_columns(pl.col("ip_late_payments").clip(0, 10).alias("late_payments_capped"))
    .group_by("late_payments_capped").agg(pl.len().alias("n"), pl.col("TARGET").mean().alias("default_rate"))
    .sort("late_payments_capped"))

In [ ]:
(mar.with_columns((pl.col("bureau_total_delinquent_months") > 0).alias("any_bureau_delinquency"))
    .group_by("any_bureau_delinquency").agg(pl.len().alias("n"), pl.col("TARGET").mean().alias("default_rate")))

## Portfolio stability (PSI, baseline = older 70% vs recent 30%)

In [ ]:
pl.read_parquet(f"{P}/portfolio_stability_report.parquet").sort("psi_score", descending=True).head(10)

## Roll-rate matrix (monthly transition shares; change the source filter for credit card / POS)

In [ ]:
(pl.read_parquet(f"{P}/roll_rate_matrix.parquet")
   .filter(pl.col("source") == "bureau")  # also: "credit_card", "pos_cash"
   .pivot(on="to_bucket", index="from_bucket", values="roll_rate")
   .with_columns(pl.exclude("from_bucket").round(3)))

## Vintage curves: cumulative 61+ DPD rate at 12 months on book, by source and opening cohort

In [ ]:
(pl.read_parquet(f"{P}/vintage_curves.parquet")
   .filter((pl.col("mob") == 12) & (pl.col("accounts_observed") >= 1000))
   .select("source", "cohort_start_month", "accounts_observed", "cumulative_bad_rate")
   .sort("source", "cohort_start_month"))

## Diagnostic brief

In [ ]:
from IPython.display import Markdown
Markdown(open(f"{P}/diagnostic_brief.md").read())